# 노후주택 데이터로 YOLO 학습 + 등급 실증 (Colab)

AI-Hub는 **해외(코랩) IP 다운로드를 차단**하므로, 데이터 다운로드·변환은 **한국 IP의 내 Mac**에서 끝내고 여기선 **학습만** 합니다.

- 데이터셋: `567` 서울시 노후 주택 균열 데이터 (비주거용주택 = 노후 상가류, 우리 타깃)
- 학습 클래스 3종: crack / spalling / rebar

## 사전 작업 (내 Mac에서, 이미 완료)
```bash
bash data-tools/download_convert_house.sh 567 "400979,400984" 800
```
→ 생성된 `~/house_data/house_yolo.zip` (약 945MB)를 **구글 드라이브 My Drive 최상위**에 업로드.

## 이 노트북
**런타임 → 런타임 유형 변경 → GPU(T4/L4/A100)** 설정 후 위에서부터 실행.

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. 드라이브 연결 + 데이터 압축해제
`house_yolo.zip` 이 My Drive 최상위에 있다고 가정. 경로가 다르면 `ZIP_PATH` 수정. `train … | val …` 개수가 뜨면 성공.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, glob
ZIP_PATH = '/content/drive/MyDrive/house_yolo.zip'
assert os.path.exists(ZIP_PATH), f'파일 없음: {ZIP_PATH}  (드라이브 최상위에 올렸는지 확인)'
os.makedirs('/content/house_yolo', exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/house_yolo')

# zip 내부에 상위 폴더가 한 겹 있을 수 있으니 data.yaml 위치를 자동 탐색
yamls = glob.glob('/content/house_yolo/**/data.yaml', recursive=True)
assert yamls, 'data.yaml 을 찾을 수 없음'
ROOT = os.path.dirname(yamls[0])
yaml_path = os.path.join(ROOT, 'data.yaml')

# data.yaml 의 path 를 Colab 경로로 보정
lines = open(yaml_path, encoding='utf-8').read().splitlines()
open(yaml_path, 'w', encoding='utf-8').write(
    '\n'.join(f'path: {ROOT}' if ln.startswith('path:') else ln for ln in lines) + '\n')

print('데이터 루트:', ROOT)
print('train:', len(glob.glob(f'{ROOT}/images/train/*')), '| val:', len(glob.glob(f'{ROOT}/images/val/*')))
print(open(yaml_path, encoding='utf-8').read())

## 3. YOLO 학습
클래스 3종(crack/spalling/rebar). 더 가볍게: `yolo11n.pt` / 더 정확: `yolo11m.pt`.

In [ ]:
%pip install -q ultralytics
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    name='house_crack',
)

## 4. best.pt 저장 (다운로드 + 드라이브 백업)
이 `best.pt` 를 `ai-server/models/best.pt` 에 넣으면 서버가 mock → 실제 추론으로 전환됩니다.

In [ ]:
import os, shutil
from IPython.display import Image, display
run_dir = results.save_dir
res_png = os.path.join(run_dir, 'results.png')
if os.path.exists(res_png):
    display(Image(filename=res_png, width=900))
best = os.path.join(run_dir, 'weights', 'best.pt')
print('best.pt:', best)

shutil.copy(best, '/content/drive/MyDrive/house_crack_best.pt')
print('드라이브 저장: /content/drive/MyDrive/house_crack_best.pt')
from google.colab import files
files.download(best)

## 5. ★ 등급 실증 — 우수/보통/불량 vs 우리 A~E (confusion matrix)
교수님이 원하는 **'사용자 체감 최종 정확도'** 실증. 전문가 라벨 없이 데이터 등급 라벨만으로 즉시 검증.

val 이미지에 대해: 탐지 → 간이 트리아지 점수 → A~E → **우수(A·B)/보통(C)/불량(D·E)** 그룹 매핑 → `grades.json`(실제 등급)과 대조. **위험 누락률**(실제 불량을 우수/보통으로 오분류)이 핵심 안전 지표.

In [ ]:
import json, glob, os
from collections import defaultdict

SEVERITY = {'crack': 0.55, 'spalling': 0.85, 'rebar': 0.95}
W_SEV, W_DENS, W_CNT = 0.55, 0.30, 0.15
def score_to_group(score):
    if score >= 60: return '불량'   # D,E
    if score >= 40: return '보통'   # C
    return '우수'                   # A,B

grades = json.load(open(os.path.join(ROOT, 'grades.json'), encoding='utf-8'))
names = model.names
val_imgs = glob.glob(f'{ROOT}/images/val/*')

GROUPS = ['우수', '보통', '불량']
cm = defaultdict(int)
n = 0
for img in val_imgs:
    true_g = grades.get(os.path.basename(img))
    if true_g not in GROUPS:
        continue
    r = model.predict(img, conf=0.25, verbose=False)[0]
    W, H = r.orig_shape[1], r.orig_shape[0]
    area = max(W * H, 1)
    sev = dens = cnt = 0.0
    boxes = r.boxes
    if boxes is not None and len(boxes) > 0:
        for b in boxes:
            cls = names[int(b.cls)]
            sev = max(sev, SEVERITY.get(cls, 0.5))
            x1, y1, x2, y2 = b.xyxy[0].tolist()
            dens += (x2 - x1) * (y2 - y1)
        dens = min(dens / area, 1.0)
        cnt = min(len(boxes) / 10.0, 1.0)
    score = 100 * (W_SEV * sev + W_DENS * dens + W_CNT * cnt)
    pred_g = score_to_group(score)
    cm[(true_g, pred_g)] += 1
    n += 1

print(f'검증 이미지(등급 라벨 有): {n}장\n')
print('실제\\예측 |', ' | '.join(f'{g:>4}' for g in GROUPS))
for t in GROUPS:
    row = [cm[(t, p)] for p in GROUPS]
    print(f'{t:>6}   |', ' | '.join(f'{v:>4}' for v in row))

correct = sum(cm[(g, g)] for g in GROUPS)
acc = correct / n if n else 0
danger_total = sum(cm[('불량', p)] for p in GROUPS)
missed = cm[('불량', '우수')] + cm[('불량', '보통')]
miss_rate = missed / danger_total if danger_total else 0
print(f'\n등급 정확도: {acc:.1%}')
print(f'위험 누락률(실제 불량을 놓침): {miss_rate:.1%}  ← 안전상 가장 중요, 낮을수록 좋음')

## 6. 빠른 추론 시각화 (선택)

In [ ]:
import glob, os
from IPython.display import Image, display
best_model = YOLO(best)
for img in glob.glob(f'{ROOT}/images/val/*')[:3]:
    p = best_model.predict(img, save=True, conf=0.25, verbose=False)[0]
    display(Image(filename=os.path.join(p.save_dir, os.path.basename(p.path)), width=600))